# RiskPulse DeepML — 03. Feature Engineering, Splitting & Preprocessing Pipelines

**Objective**:
- Load the cleaned dataset (`data/cleaned_credit_risk_dataset.csv`).
- Separate feature matrix ($X$) and target vector ($y = \text{loan\_status}$).
- Perform a Stratified Train-Test Split (80% Train / 20% Test).
- Handle class imbalance using cost-sensitive weighting / resampling strategies.
- Build Scikit-Learn `ColumnTransformer` & `Pipeline` for numeric scaling and categorical encoding.

### 01. Environment Setup & Load Cleaned Data

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_clean = pd.read_csv("../data/cleaned_credit_risk_dataset.csv")





In [29]:
df_clean.shape[0]

31522

### 02. Feature Selection ($X$) & Target Definition ($y$)

In [56]:
# 1. Separate Features (X) and Target (y)
X = df_clean.drop(columns=['loan_status'])
y = df_clean['loan_status']

### 03. Stratified Train-Test Split

In [31]:
from sklearn.model_selection import train_test_split

X_train, y_train, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [57]:
# 2. Extract feature column lists directly from X
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"{numeric_col[2]}, {categorical_col[2]}")

person_emp_length, loan_grade


C:\Users\iamkh\AppData\Local\Temp\ipykernel_21276\3031918871.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()


### 04. Class Imbalance Inspection & Handling Strategy

In [37]:
# Class-weight method 
en, de=np.bincount(y_train)

scale_weight = en / de
scale_weight

np.float64(3.6312213039485766)

### 05. Preprocessing Pipelines & `ColumnTransformer` Setup

In [38]:
#Logistic Regression : impute + SCALE Numeric, impute + One-Hot-Encoding
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [58]:
#Logistic Regression : impute + SCALE Numeric, impute + One-Hot-Encoding

preprocessor_lr = ColumnTransformer(
    transformers= [
        # pipeline for numeric value
        (
            'num', Pipeline(
                [
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler())
                ]
            ), num_cols),
         # pipeline for categorical value

         ('cat', Pipeline(
            [
                ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]), cat_cols)   
    ]
)

In [59]:
#XGBoost: Impute numeric only (No Scaling), impute + One - Hot Encoding

preprocessor_xg = ColumnTransformer(
    transformers=[
        #Pipeline - for numerical cols
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), num_cols),

        #Pipeline - for Categorical cols.
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy="constant", fill_value="Missing")),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols)
    ])

     


### 6. Preprocessing Verification & Feature Summary

#### Stratified Cross Validation

In [60]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits = 5, shuffle= True, random_state= 42)

scoring = {'roc_auc': 'roc_auc', 'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1'}

In [61]:
# Logitic Regression
from sklearn.linear_model import LogisticRegression
lr_cv_pipeline = Pipeline([
    ('Preprocessor', preprocessor_lr),
    ("Classifier", LogisticRegression(max_iter= 1000, class_weight= 'balanced', random_state=42))
])

In [62]:
# Xgboost Model
from xgboost import XGBClassifier

xgb_cv_pipeline= Pipeline([
    ('Preprocessor', preprocessor_xg),
    ('Classifier', XGBClassifier(
        n_estimator= 300, max_depth= 5, learning_rate= 0.1,
        scale_pos_weight= scale_weight, random_state=42
    ))
])

In [63]:
for cv_name, pipe in [("Logistic Regression", lr_cv_pipeline),
                      ("XGBoost", xgb_cv_pipeline)]:

                      cv_results = cross_validate(pipe, X_train , y_train, cv=cv, scoring=scoring, n_jobs=-1)

                      print(cv_name)
                      print(f"ROC-AUC : {cv_results['test_roc_auc'].mean()}")
                      print(f"Accuracy : {cv_results['test_accuracy'].mean()}")
                      print(f"Precision : {cv_results['test_precision'].mean()}")
                      print(f"Recall : {cv_results['test_recall'].mean()}")
                      print(f"F1 : {cv_results['test_f1'].mean()}")
                      print()
     

Logistic Regression
ROC-AUC : 0.8705112830525306
Accuracy : 0.8115949542892269
Precision : 0.5448226008889892
Recall : 0.7781450872359963
F1 : 0.6408013995933108

XGBoost
ROC-AUC : 0.9393848243386111
Accuracy : 0.9086724038455642
Precision : 0.7913354492271174
Recall : 0.7856749311294765
F1 : 0.7880291923045915

